# Heritage Restore — HYPIR Colab Inference Server

Runs HYPIR (SD2.1-based generative restoration) on a free Colab T4 GPU and exposes it as an HTTP API via a Cloudflare quick tunnel, so your local backend can call it as the `generative_enhance` restoration stage.

**Before you run anything:** Runtime → Change runtime type → **T4 GPU**.

**Read this before using it:**
- Keep this tab open and active while testing. Idle Colab sessions disconnect, and the public URL dies with the session.
- This is a **dev/experimentation path only**. Google's free-tier Colab terms disallow using a runtime primarily as a bypassed web server on sessions with no positive compute-unit balance, and such sessions can be terminated without warning. Don't rely on this for a live demo or interview — keep a hosted-API fallback in your backend for anything user-facing.
- HYPIR (XPixelGroup) is released for **non-commercial use only**.
- The public URL changes every time this notebook restarts — you'll need to copy the new one into your backend's `.env` each time.

## 0. Check the GPU

In [ ]:
!nvidia-smi

## 1. Install HYPIR

In [ ]:
%cd /content
!git clone https://github.com/XPixelGroup/HYPIR.git
%cd HYPIR
!pip install -q -r requirements.txt
!pip install -q fastapi uvicorn python-multipart

## 2. Download pretrained weights

HYPIR needs two things: the Stable Diffusion 2.1 base model, and the HYPIR LoRA weights. If a download fails with a 401/403, open the model page on huggingface.co, click "Agree" on any license terms shown, then re-run this cell.

In [ ]:
import os
from getpass import getpass
from huggingface_hub import login, snapshot_download, hf_hub_download

token = getpass("Hugging Face token (hf_...), or press Enter to skip if not needed: ")
if token.strip():
    login(token=token.strip())

BASE_MODEL_DIR = "/content/models/stable-diffusion-2-1-base"
LORA_WEIGHT_PATH = "/content/models/HYPIR_sd2.pth"
os.makedirs("/content/models", exist_ok=True)

print("Downloading Stable Diffusion 2.1 base model (several minutes, ~5GB)...")
snapshot_download(
    repo_id="stabilityai/stable-diffusion-2-1-base",
    local_dir=BASE_MODEL_DIR,
    allow_patterns=[
        "model_index.json",
        "scheduler/*",
        "tokenizer/*",
        "text_encoder/config.json",
        "text_encoder/model.safetensors",
        "unet/config.json",
        "unet/diffusion_pytorch_model.safetensors",
        "vae/config.json",
        "vae/diffusion_pytorch_model.safetensors",
    ],
)

print("Downloading HYPIR LoRA weights...")
hf_hub_download(
    repo_id="lxq007/HYPIR",
    filename="HYPIR_sd2.pth",
    local_dir="/content/models",
)

print("Done.")
print("Base model dir:", BASE_MODEL_DIR)
print("LoRA weights:", LORA_WEIGHT_PATH)

## 3. Set a shared-secret API key

Your backend must send this exact key in the `X-API-Key` header on every request. Pick any random string — don't reuse a real password. You'll need to copy this same value into your backend's `.env` as `COLAB_HYPIR_API_KEY`.

In [ ]:
from getpass import getpass
API_KEY = getpass("Set a shared-secret API key for this session: ")
print("API key set (hidden). Remember to copy the same value into your backend's .env.")

## 4. Write the FastAPI server

This wraps HYPIR's `test.py` the same way a typical restoration backend wraps a model's own CLI script: writes the input image to a temp directory, invokes the documented command, reads back the output file. Arguments match the official HYPIR README inference example.

In [ ]:
%%writefile /content/server.py
import base64
import os
import shutil
import subprocess
import tempfile
import time
import uuid

import torch
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel

API_KEY = os.environ["HYPIR_API_KEY"]
HYPIR_DIR = "/content/HYPIR"
BASE_MODEL_DIR = "/content/models/stable-diffusion-2-1-base"
LORA_WEIGHT_PATH = "/content/models/HYPIR_sd2.pth"

LORA_MODULES = (
    "to_k,to_q,to_v,to_out.0,conv,conv1,conv2,conv_shortcut,"
    "conv_out,proj_in,proj_out,ff.net.2,ff.net.0.proj"
)

app = FastAPI()
_model_ready = os.path.isfile(LORA_WEIGHT_PATH) and os.path.isdir(BASE_MODEL_DIR)


class EnhanceRequest(BaseModel):
    image_base64: str
    patch_size: int = 512
    stride: int = 256
    prompt: str = ""


def check_key(x_api_key: str):
    if x_api_key != API_KEY:
        raise HTTPException(status_code=401, detail="Invalid API key")


@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": _model_ready}


@app.post("/enhance")
def enhance(req: EnhanceRequest, x_api_key: str = Header(default="")):
    check_key(x_api_key)
    if not _model_ready:
        raise HTTPException(status_code=503, detail="Model weights not found")

    work_dir = tempfile.mkdtemp(prefix="hypir_req_")
    lq_dir = os.path.join(work_dir, "lq")
    txt_dir = os.path.join(work_dir, "prompt")
    out_dir = os.path.join(work_dir, "out")
    for d in (lq_dir, txt_dir, out_dir):
        os.makedirs(d, exist_ok=True)

    img_name = f"{uuid.uuid4().hex}.png"
    img_path = os.path.join(lq_dir, img_name)
    with open(img_path, "wb") as f:
        f.write(base64.b64decode(req.image_base64))

    txt_path = os.path.join(txt_dir, img_name.replace(".png", ".txt"))
    with open(txt_path, "w") as f:
        f.write(req.prompt)

    torch.cuda.reset_peak_memory_stats()
    start = time.time()

    cmd = [
        "python", "test.py",
        "--base_model_type", "sd2",
        "--base_model_path", BASE_MODEL_DIR,
        "--model_t", "200",
        "--coeff_t", "200",
        "--lora_rank", "256",
        "--lora_modules", LORA_MODULES,
        "--weight_path", LORA_WEIGHT_PATH,
        "--patch_size", str(req.patch_size),
        "--stride", str(req.stride),
        "--lq_dir", lq_dir,
        "--scale_by", "factor",
        "--upscale", "1",
        "--txt_dir", txt_dir,
        "--output_dir", out_dir,
        "--seed", "231",
        "--device", "cuda",
    ]
    result = subprocess.run(
        cmd, cwd=HYPIR_DIR, capture_output=True, text=True, timeout=600
    )
    elapsed_ms = int((time.time() - start) * 1000)
    peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

    if result.returncode != 0:
        shutil.rmtree(work_dir, ignore_errors=True)
        raise HTTPException(
            status_code=500,
            detail=f"HYPIR failed: {result.stderr[-2000:]}",
        )

    result_dir = os.path.join(out_dir, "result")
    if not os.path.exists(result_dir):
        shutil.rmtree(work_dir, ignore_errors=True)
        raise HTTPException(status_code=500, detail="HYPIR produced no result directory")
    out_files = [f for f in os.listdir(result_dir) if f.lower().endswith((".png", ".jpg"))]
    if not out_files:
        shutil.rmtree(work_dir, ignore_errors=True)
        err_msg = f"HYPIR produced no output file.\nSTDOUT: {result.stdout[-1500:]}\nSTDERR: {result.stderr[-1500:]}"
        raise HTTPException(status_code=500, detail=err_msg)

    out_path = os.path.join(result_dir, out_files[0])
    with open(out_path, "rb") as f:
        out_b64 = base64.b64encode(f.read()).decode("utf-8")

    shutil.rmtree(work_dir, ignore_errors=True)

    return {
        "image_base64": out_b64,
        "processing_time_ms": elapsed_ms,
        "peak_vram_mb": round(peak_vram_mb, 1),
        "patch_size": req.patch_size,
        "stride": req.stride,
    }

## 5. Start the server

In [ ]:
import os, subprocess, threading, time

os.environ["HYPIR_API_KEY"] = API_KEY

server_proc = subprocess.Popen(
    ["uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

def stream_server_logs():
    for line in server_proc.stdout:
        print("[server]", line, end="")

threading.Thread(target=stream_server_logs, daemon=True).start()
time.sleep(5)
print("Server starting on port 8000...")

## 6. Start the Cloudflare quick tunnel

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

import subprocess, re, threading

tunnel_proc = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
url_pattern = re.compile(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com")

def stream_tunnel_logs():
    global public_url
    for line in tunnel_proc.stdout:
        match = url_pattern.search(line)
        if match and public_url is None:
            public_url = match.group(0)
            print("=" * 60)
            print("COLAB_HYPIR_URL:", public_url)
            print("=" * 60)
            print("Copy this into your backend .env as COLAB_HYPIR_URL")
            print("This URL changes every time this notebook restarts.")

threading.Thread(target=stream_tunnel_logs, daemon=True).start()
print("Waiting for tunnel URL (usually a few seconds)...")

## 7. Test it from inside the notebook

In [ ]:
import requests, time

for _ in range(30):
    if public_url:
        break
    time.sleep(1)

if not public_url:
    print("Tunnel URL not ready yet. Re-run this cell, or check the logs above for errors.")
else:
    resp = requests.get(
        f"{public_url}/health",
        headers={"X-API-Key": API_KEY},
        timeout=15,
    )
    print("Health check:", resp.status_code, resp.json())

## Keeping this alive

- This notebook must stay open with an active browser tab for the session and tunnel to keep running.
- If the runtime restarts, re-run every cell from the top — the public URL will be different, so update `COLAB_HYPIR_URL` in your backend's `.env` again.
- To test your backend's fallback behavior deliberately: Runtime → Interrupt execution, or just close this tab, then confirm your backend's health-check-then-fallback logic kicks in cleanly.